# NeuralGCM Forecast Reliability — Environment Setup & First Inference

This notebook sets up the Colab environment and reproduces a basic NeuralGCM inference to verify everything works.

**Runtime**: Go to `Runtime > Change runtime type` and select **T4 GPU** (Colab Pro).

## 1. Install Dependencies

Colab comes with JAX pre-installed (with GPU support). We just need to install
neuralgcm and its dependencies on top. The `!pip install` commands run in the
Colab VM's Python environment — there's no need for conda or virtualenvs.

In [ ]:
# Install neuralgcm and dependencies (~2-3 min on first run)
# neuralgcm pulls in: dinosaur (dynamical core), dm-haiku, gin-config, optax
# gcsfs is needed to load checkpoints and ERA5 data from Google Cloud Storage
!pip install -q neuralgcm dinosaur gcsfs

In [ ]:
# Verify GPU is available and JAX can see it
import jax
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
assert any(d.platform == 'gpu' for d in jax.devices()), (
    "No GPU found! Go to Runtime > Change runtime type > A100 GPU"
)

## 2. Imports

In [ ]:
import gcsfs
import jax
import numpy as np
import pickle
import xarray

from dinosaur import horizontal_interpolation
from dinosaur import spherical_harmonic
from dinosaur import xarray_utils
import neuralgcm

print(f"neuralgcm version: {neuralgcm.__version__}")

## 3. Load Pretrained Model

Start with the 2.8° deterministic model for fast prototyping (~2.5s per 10-day forecast).
Switch to `stochastic_1_4_deg` for the real experiments.

In [ ]:
model_name = 'v1/deterministic_2_8_deg.pkl'  # fast; switch to 'v1/stochastic_1_4_deg.pkl' later

gcs = gcsfs.GCSFileSystem(token='anon')
with gcs.open(f'gs://neuralgcm/models/{model_name}', 'rb') as f:
    ckpt = pickle.load(f)

model = neuralgcm.PressureLevelModel.from_checkpoint(ckpt)
print(f"Model loaded: {model_name}")
print(f"Input variables: {model.input_variables}")
print(f"Forcing variables: {model.forcing_variables}")

## 4. Load ERA5 Data

ERA5 reanalysis is hosted on Google Cloud Storage in Zarr format. We load a
small time slice and regrid to the model's native resolution.

In [ ]:
era5_path = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
full_era5 = xarray.open_zarr(
    era5_path, chunks=None, storage_options=dict(token='anon')
)

# Select a short demo period (4 days from the 2020 holdout year)
demo_start_time = '2020-02-14'
demo_end_time = '2020-02-18'
data_inner_steps = 24  # sample every 24 hours

sliced_era5 = (
    full_era5
    [model.input_variables + model.forcing_variables]
    .pipe(
        xarray_utils.selective_temporal_shift,
        variables=model.forcing_variables,
        time_shift='24 hours',
    )
    .sel(time=slice(demo_start_time, demo_end_time, data_inner_steps))
    .compute()
)
print(f"ERA5 slice loaded: {dict(sliced_era5.sizes)}")

In [ ]:
# Regrid ERA5 (0.25°) to model's native resolution
era5_grid = spherical_harmonic.Grid(
    latitude_nodes=full_era5.sizes['latitude'],
    longitude_nodes=full_era5.sizes['longitude'],
    latitude_spacing=xarray_utils.infer_latitude_spacing(full_era5.latitude),
    longitude_offset=xarray_utils.infer_longitude_offset(full_era5.longitude),
)
regridder = horizontal_interpolation.ConservativeRegridder(
    era5_grid, model.data_coords.horizontal, skipna=True
)
eval_era5 = xarray_utils.regrid(sliced_era5, regridder)
eval_era5 = xarray_utils.fill_nan_with_nearest(eval_era5)
print(f"Regridded to model resolution: {dict(eval_era5.sizes)}")

## 5. Run Inference (4-day forecast)

In [ ]:
import time

inner_steps = 24   # save outputs every 24 hours
outer_steps = 4    # 4 days total
timedelta = np.timedelta64(1, 'h') * inner_steps
times = np.arange(outer_steps) * inner_steps  # time axis in hours

# Initialize model state from first ERA5 timestep
inputs = model.inputs_from_xarray(eval_era5.isel(time=0))
input_forcings = model.forcings_from_xarray(eval_era5.isel(time=0))
rng_key = jax.random.key(42)
initial_state = model.encode(inputs, input_forcings, rng_key)

# Use persistence forcing (constant SST and sea ice)
all_forcings = model.forcings_from_xarray(eval_era5.head(time=1))

# Run forecast and time it
t0 = time.time()
final_state, predictions = model.unroll(
    initial_state,
    all_forcings,
    steps=outer_steps,
    timedelta=timedelta,
    start_with_input=True,
)
predictions_ds = model.data_to_xarray(predictions, times=times)
elapsed = time.time() - t0
print(f"Forecast complete: {outer_steps} days in {elapsed:.1f}s")
print(f"Prediction variables: {list(predictions_ds.data_vars)}")

## 6. Compare Forecast vs ERA5 (Sanity Check)

In [ ]:
# Build ERA5 target trajectory for comparison
target_trajectory = model.inputs_from_xarray(
    eval_era5
    .thin(time=(inner_steps // data_inner_steps))
    .isel(time=slice(outer_steps))
)
target_ds = model.data_to_xarray(target_trajectory, times=times)

# Combine for side-by-side plotting
combined_ds = xarray.concat([target_ds, predictions_ds], 'model')
combined_ds.coords['model'] = ['ERA5', 'NeuralGCM']

# Plot specific humidity at 850 hPa
combined_ds.specific_humidity.sel(level=850).plot(
    x='longitude', y='latitude', row='time', col='model',
    robust=True, aspect=2, size=2
);

In [ ]:
# Quick RMSE sanity check
import numpy as np

for var in ['temperature', 'geopotential', 'specific_humidity']:
    if var in predictions_ds and var in target_ds:
        diff = predictions_ds[var] - target_ds[var]
        # Area-weighted RMSE at final timestep
        weights = np.cos(np.deg2rad(predictions_ds.latitude))
        rmse = float(np.sqrt((diff.isel(time=-1) ** 2).weighted(weights).mean()).values)
        print(f"{var} RMSE (day {outer_steps}): {rmse:.4f}")

---

## 7. Stochastic Ensemble Inference (NeuralGCM-ENS 1.4°)

The stochastic model generates different ensemble members by injecting noise
during `encode()`. Each unique `rng_key` produces a distinct member.

The Nature paper uses 50-member ensembles. We start with a small ensemble
(N=4) for prototyping, then scale up.

In [ ]:
# Load the stochastic 1.4° model
stoch_model_name = 'v1/stochastic_1_4_deg.pkl'

with gcs.open(f'gs://neuralgcm/models/{stoch_model_name}', 'rb') as f:
    stoch_ckpt = pickle.load(f)

stoch_model = neuralgcm.PressureLevelModel.from_checkpoint(stoch_ckpt)
print(f"Stochastic model loaded: {stoch_model_name}")
print(f"Model resolution: {stoch_model.data_coords}")
print(f"Timestep: {stoch_model.timestep}")

In [ ]:
# Regrid ERA5 to the stochastic model's 1.4° resolution
stoch_regridder = horizontal_interpolation.ConservativeRegridder(
    era5_grid, stoch_model.data_coords.horizontal, skipna=True
)
stoch_era5 = xarray_utils.regrid(sliced_era5, stoch_regridder)
stoch_era5 = xarray_utils.fill_nan_with_nearest(stoch_era5)
print(f"Regridded to 1.4°: {dict(stoch_era5.sizes)}")

### Run ensemble forecast

Each member gets a different `rng_key` passed to `encode()`, which injects
different noise into the learned encoder and physics module. The `unroll` call
itself is deterministic given the initial state.

In [ ]:
n_ensemble = 4  # start small; scale to 50 for real experiments
inner_steps = 24
outer_steps = 4
timedelta = np.timedelta64(1, 'h') * inner_steps
times = np.arange(outer_steps) * inner_steps

stoch_inputs = stoch_model.inputs_from_xarray(stoch_era5.isel(time=0))
stoch_forcings = stoch_model.forcings_from_xarray(stoch_era5.head(time=1))

ensemble_predictions = []
t0 = time.time()

for member in range(n_ensemble):
    rng_key = jax.random.key(member)
    state = stoch_model.encode(stoch_inputs, stoch_forcings, rng_key)
    _, preds = stoch_model.unroll(
        state, stoch_forcings,
        steps=outer_steps, timedelta=timedelta, start_with_input=True,
    )
    pred_ds = stoch_model.data_to_xarray(preds, times=times)
    ensemble_predictions.append(pred_ds)
    print(f"  Member {member} done")

elapsed = time.time() - t0
print(f"\nEnsemble ({n_ensemble} members, {outer_steps} days) in {elapsed:.1f}s")
print(f"Per member: {elapsed/n_ensemble:.1f}s")

In [ ]:
# Stack ensemble into a single dataset with an 'ensemble' dimension
ensemble_ds = xarray.concat(ensemble_predictions, dim='member')
ensemble_ds.coords['member'] = np.arange(n_ensemble)
print(f"Ensemble dataset: {dict(ensemble_ds.sizes)}")

### Ensemble statistics: mean, spread, spread-skill ratio

In [ ]:
# Build ERA5 target at 1.4° resolution for comparison
stoch_target = stoch_model.inputs_from_xarray(
    stoch_era5
    .thin(time=(inner_steps // data_inner_steps))
    .isel(time=slice(outer_steps))
)
target_ds_14 = stoch_model.data_to_xarray(stoch_target, times=times)

# Ensemble mean and spread (std across members)
ens_mean = ensemble_ds.mean(dim='member')
ens_spread = ensemble_ds.std(dim='member')

# RMSE of ensemble mean vs ERA5 (area-weighted)
weights = np.cos(np.deg2rad(ensemble_ds.latitude))

print("Variable       | RMSE (ens mean) | Spread (mean) | Spread/Skill")
print("-" * 70)
for var in ['temperature', 'geopotential', 'specific_humidity']:
    if var not in ens_mean:
        continue
    error = ens_mean[var] - target_ds_14[var]
    # RMSE at final timestep, averaged over levels
    rmse = float(np.sqrt((error.isel(time=-1) ** 2).weighted(weights).mean()).values)
    spread = float(ens_spread[var].isel(time=-1).weighted(weights).mean().values)
    ratio = spread / rmse if rmse > 0 else float('nan')
    print(f"{var:<16} | {rmse:>14.4f}  | {spread:>12.4f}  | {ratio:.3f}")

### Visualize ensemble spread

Plot ensemble members overlaid to see the spread growing with lead time.

In [ ]:
import matplotlib.pyplot as plt

var = 'temperature'
level = 850
day = -1  # final timestep

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Panel 1: Ensemble mean
ens_mean[var].sel(level=level).isel(time=day).plot(
    ax=axes[0], x='longitude', y='latitude', robust=True
)
axes[0].set_title(f'Ensemble Mean — {var} {level}hPa (day {outer_steps})')

# Panel 2: Ensemble spread (std)
ens_spread[var].sel(level=level).isel(time=day).plot(
    ax=axes[1], x='longitude', y='latitude', robust=True, cmap='Reds'
)
axes[1].set_title(f'Ensemble Spread (std) — {var} {level}hPa (day {outer_steps})')

# Panel 3: ERA5 truth
target_ds_14[var].sel(level=level).isel(time=day).plot(
    ax=axes[2], x='longitude', y='latitude', robust=True
)
axes[2].set_title(f'ERA5 — {var} {level}hPa (day {outer_steps})')

plt.tight_layout()
plt.show()

### Spread growth over lead time

The spread should grow with forecast lead time as uncertainty increases.
A well-calibrated ensemble has spread/skill ~ 1.0 at all lead times.

In [ ]:
var = 'temperature'
level = 500

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rmse_by_time = []
spread_by_time = []
for t in range(outer_steps):
    error = ens_mean[var].sel(level=level).isel(time=t) - target_ds_14[var].sel(level=level).isel(time=t)
    rmse_t = float(np.sqrt((error ** 2).weighted(weights).mean()).values)
    spread_t = float(ens_spread[var].sel(level=level).isel(time=t).weighted(weights).mean().values)
    rmse_by_time.append(rmse_t)
    spread_by_time.append(spread_t)

days = np.arange(1, outer_steps + 1)

axes[0].plot(days, rmse_by_time, 'o-', label='RMSE (ens mean)')
axes[0].plot(days, spread_by_time, 's-', label='Spread (std)')
axes[0].set_xlabel('Lead time (days)')
axes[0].set_ylabel(f'{var} {level}hPa')
axes[0].legend()
axes[0].set_title('RMSE vs Spread')

ratio = [s / r if r > 0 else float('nan') for s, r in zip(spread_by_time, rmse_by_time)]
axes[1].plot(days, ratio, 'o-', color='green')
axes[1].axhline(1.0, color='gray', linestyle='--', label='Perfect calibration')
axes[1].set_xlabel('Lead time (days)')
axes[1].set_ylabel('Spread / Skill')
axes[1].legend()
axes[1].set_title('Spread-Skill Ratio')

# Panel 3: spread vs RMSE scatter
axes[2].scatter(rmse_by_time, spread_by_time, c=days, cmap='viridis', s=80, zorder=3)
max_val = max(max(rmse_by_time), max(spread_by_time)) * 1.1
axes[2].plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='1:1')
axes[2].set_xlabel('RMSE')
axes[2].set_ylabel('Spread')
axes[2].legend()
axes[2].set_title('Spread vs Skill')

plt.tight_layout()
plt.show()

print(f"\nNote: with only {n_ensemble} members, spread estimates are noisy.")
print("Scale to 50 members for reliable statistics.")

## Next Steps

1. **Scale ensemble**: Increase `n_ensemble` to 50 for publication-quality statistics
2. **Extend forecast**: Run 10-15 day forecasts (`outer_steps=15`) to see spread-skill divergence at longer lead times
3. **Multiple init times**: Loop over WeatherBench2 2020 holdout initial conditions for robust evaluation
4. **Implement CRPS**: Add proper probabilistic metrics (Week 2-3 of project plan)